# 104 - Backends and Precision

prysm's decoupled design lends itself well to any library that looks
like numpy.  It is also internally written around an unusually syntax:

```python
from prysm.mathops import np
```

This is not a simple re-export of numpy with a compile time or environment
variable or other way to replace it with something else.  Instead, we have
a tiny class `BackendShim` which is made to allow libraries to be swapped
_at any time_.  You can do, for example:

```python
from prysm.mathops import np

import torch.np as tnp

np._srcmodule = tnp
```

and then you have swapped to pytorch.  There are convenience functions:

- `mathops.set_backend_to_cupy`
- `mathops.set_backend_to_pytorch`
- `mathops.set_backend_to_pytorch`
- `mathops.set_fft_backend_to_mkl_fft`
- `mathops.set_backend_to_defaults`

which do this for all of the modules (np, scipy.ndimage, scipy.fft, etc).

The second global configuration is `conf.config` which has a controller
for the precision with which computations are done It defaults to float64.  You
can set it with a dtype or an integer bit-depth (`16`, `32`, `64`); the matching
complex dtype is derived automatically and exposed as `config.precision_complex`.

In [ ]:
import numpy as np
from prysm.conf import config
from prysm.coordinates import make_xy_grid

In [ ]:
print('default real   :', config.precision)
print('default complex :', config.precision_complex)

x, y = make_xy_grid(64, diameter=2)
print('grid dtype      :', x.dtype)

Drop to float32 and the same grid is built in single precision.  Most
calculations do just fine in 32-bit, even most coronagraph and
wavefront sensing and control models.  Many GPUs are 32 to 64 times slower
at 64-bit math compared to 32-bit math.  Using the cupy backend and
`config.precision = 32`, it is common to see speedups on the order of 10-100x
with zero loss of accuracy and almost zero lines of code you have to change in
your program.  The only exception is around I/O and reading data from disk,
where you will have to run things through `np.asarray()` and when plotting when
you will need to pull things back to vanilla numpy, although this will improve
in a future release when using prysm's built in plots.

In [ ]:
config.precision = 32
x32, _ = make_xy_grid(64, diameter=2)
print('precision       :', config.precision)
print('complex         :', config.precision_complex)
print('grid dtype      :', x32.dtype)

config.precision = 64   # restore the default for the rest of the notebook
print('restored        :', config.precision)

This completes the foundational college.  At this point you should have a general grasp of how prysm works, and you can dive in to any of the topical colleges:

TODO LIST OF TOPICAL COLLEGES